# Export CLIP ViT-B/32 + Florence-2-base for on-device iOS

Exports two models to CoreML `.mlpackage` for use in the iOS app:

- **CLIP ViT-B/32** (`openai/clip-vit-base-patch32`) &mdash; image + text encoders for on-device **image retrieval** (embed a library of photos, embed a text query, rank by cosine similarity).
- **Florence-2-base** (`microsoft/Florence-2-base`) &mdash; an image encoder + autoregressive text decoder for on-device **image captioning**.

Both are validated against the original PyTorch models (cosine similarity of exported vs. reference outputs) before being written to `exported_models/`.

## Why Florence-2 needs a different export strategy than CLIP

CLIP's image/text encoders are plain feed-forward transformers &mdash; a single `torch.jit.trace` + `coremltools.convert` gets you a working model.

Florence-2 is a full encoder-decoder (image tokens + task-prompt tokens &rarr; transformer encoder &rarr; autoregressive transformer decoder &rarr; vocabulary logits), and `model.generate()` is a Python control-flow loop with a growing KV-cache &mdash; none of that traces directly. The strategy used here:

1. **Bake the captioning task into the export.** Florence-2's processor maps a task token like `"<CAPTION>"` to a *fixed* natural-language prompt (e.g. `"What does the image describe?"`) whose token embedding never depends on the image. That prompt is embedded once at export time and folded into the graph as a constant, so the exported encoder's only runtime input is the image.
2. **Export two models, not one:** `FlorenceEncoder` (image &rarr; encoder hidden states, computed once per image) and `FlorenceDecoderStep` (encoder hidden states + tokens-so-far &rarr; next-token logits). iOS calls the encoder once, then calls the decoder step repeatedly in a greedy-decoding loop, exactly like `model.generate()` does internally.
3. **No KV-cache.** The decoder step recomputes attention over the whole generated-so-far sequence at every step instead of caching keys/values. This is O(n&sup2;) instead of O(n) in sequence length, but captions are short (a few dozen tokens) and the decoder is small (6 layers, d_model 768), so it's cheap &mdash; and it avoids the substantial extra complexity of exporting a stateful KV-cache CoreML model. A worthwhile trade-off for a trial/prototype export; revisit with CoreML's stateful-model API if decode latency ever matters.
4. **`torch.export` instead of `torch.jit.trace`.** Florence-2's vision tower (DaViT) and its BART-style attention/position-embedding helpers build masks and position ids from a traced tensor's `.shape` (e.g. `torch.full((seq_len, seq_len), ...)`). `coremltools` 9.0's TorchScript-based converter cannot convert the resulting `aten::Int` op ("only 0-dimensional arrays can be converted to Python scalars") even though the shape is a compile-time constant for our fixed 768&times;768 input. Exporting via `torch.export.export(...).run_decompositions({})` and feeding the resulting `ExportedProgram` to `coremltools.convert` sidesteps this entirely and lets us call the model's own methods unmodified (no need to hand-reimplement DaViT or the BART decoder).

For CLIP, plain `torch.jit.trace` is used since it's simpler and the model is small; one spot (the text encoder's EOS-token pooling) needed a small rewrite for the same class of tracer issue &mdash; explained inline below.


In [1]:
import glob
import json
from pathlib import Path

import coremltools as ct
import numpy as np
import torch
from PIL import Image

OUTPUT_DIR = Path.cwd() / "exported_models"
OUTPUT_DIR.mkdir(exist_ok=True)

IMAGES_DIR = Path.cwd().parent / "images"
IMAGE_PATHS = sorted(
    p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png")
)
print(f"Found {len(IMAGE_PATHS)} sample images: {[p.name for p in IMAGE_PATHS]}")

torch.manual_seed(0)
print("torch:", torch.__version__, " coremltools:", ct.__version__)


Torch version 2.14.0 has not been tested with coremltools. You may run into unexpected errors. Torch 2.7.0 is the most recent version that has been tested.


Found 3 sample images: ['IMG_4141.jpg', 'IMG_4277.JPG', 'IMG_4302.jpg']
torch: 2.14.0  coremltools: 9.0


## Part 1 — CLIP ViT-B/32 (image retrieval)

Two CoreML models: an **image encoder** (`pixel_values` &rarr; 512-d L2-normalized embedding) and a **text encoder** (`input_ids` + `attention_mask` &rarr; 512-d L2-normalized embedding). On-device retrieval is then just: embed every photo once with the image encoder, embed the query with the text encoder, rank by dot product.

In [2]:
from transformers import CLIPModel, CLIPProcessor

CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
for p in clip_model.parameters():
    p.requires_grad_(False)

CLIP_IMAGE_SIZE = clip_model.config.vision_config.image_size
CLIP_SEQ_LEN = clip_model.config.text_config.max_position_embeddings  # 77
print(f"image_size={CLIP_IMAGE_SIZE} projection_dim={clip_model.config.projection_dim} "
      f"text_context_length={CLIP_SEQ_LEN}")


def clip_preprocess_pil(pil_image, size=CLIP_IMAGE_SIZE):
    # Resize-shortest-edge + center-crop, matching CLIPImageProcessor's default
    # resize/crop (everything except normalization, which CoreML bakes in via
    # ImageType scale/bias). A plain `.resize((size, size))` distorts the aspect
    # ratio and changes image content, not just precision -- that would make a
    # PyTorch-vs-CoreML comparison confound real numerical fidelity with a
    # self-inflicted difference in what each model is even looking at.
    w, h = pil_image.size
    short, long_ = (w, h) if w <= h else (h, w)
    new_short = size
    new_long = int(round(long_ * (new_short / short)))
    new_w, new_h = (new_short, new_long) if w <= h else (new_long, new_short)
    resized = pil_image.resize((new_w, new_h), Image.BICUBIC)
    left = (new_w - size) // 2
    top = (new_h - size) // 2
    return resized.crop((left, top, left + size, top + size))


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


image_size=224 projection_dim=512 text_context_length=77


### Image encoder

`ct.ImageType` with `scale`/`bias` bakes CLIP's per-channel normalization into the CoreML model, so the iOS side just hands it an RGB `CVPixelBuffer` resized to 224&times;224 &mdash; no manual normalization needed in Swift.

In [3]:
class ClipImageEncoder(torch.nn.Module):
    def __init__(self, clip):
        super().__init__()
        self.clip = clip

    def forward(self, pixel_values):
        feats = self.clip.get_image_features(pixel_values=pixel_values)
        return feats / feats.norm(p=2, dim=-1, keepdim=True)


clip_image_encoder = ClipImageEncoder(clip_model).eval()

dummy_pixels = torch.rand(1, 3, CLIP_IMAGE_SIZE, CLIP_IMAGE_SIZE)
with torch.no_grad():
    traced_image_encoder = torch.jit.trace(clip_image_encoder, (dummy_pixels,))

image_mean = clip_processor.image_processor.image_mean
image_std = clip_processor.image_processor.image_std
# CoreML's ImageType applies `pixel * scale + bias` per channel with a single shared
# scale, so fold the (near-identical) per-channel stds into one average scale.
image_scale = 1.0 / (255.0 * float(np.mean(image_std)))
image_bias = [-m / s for m, s in zip(image_mean, image_std)]

clip_image_model = ct.convert(
    traced_image_encoder,
    inputs=[
        ct.ImageType(
            name="pixel_values",
            shape=dummy_pixels.shape,
            scale=image_scale,
            bias=image_bias,
            color_layout=ct.colorlayout.RGB,
        )
    ],
    outputs=[ct.TensorType(name="image_embeds")],
    convert_to="mlprogram",
    minimum_deployment_target=ct.target.iOS16,
    compute_precision=ct.precision.FLOAT16,
)
clip_image_model.save(str(OUTPUT_DIR / "CLIPImageEncoder.mlpackage"))
print("saved CLIPImageEncoder.mlpackage")


/Users/rafiabhista/Documents/code/apple/c3_agentic_ai/app-v1/trial_models/.venv/lib/python3.12/site-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/rafiabhista/Documents/code/apple/c3_agentic_ai/app-v1/trial_models/.venv/lib/python3.12/site-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


/Users/rafiabhista/Documents/code/apple/c3_agentic_ai/app-v1/trial_models/.venv/lib/python3.12/site-packages/transformers/models/clip/modeling_clip.py:243: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if not interpolate_pos_encoding and (height != self.image_size or width != self.image_size):


Converting PyTorch Frontend ==> MIL Ops:   0%|          | 0/431 [00:00<?, ? ops/s]

Converting PyTorch Frontend ==> MIL Ops: 100%|█████████▉| 430/431 [00:00<00:00, 12458.39 ops/s]

Running MIL frontend_pytorch pipeline:   0%|          | 0/5 [00:00<?, ? passes/s]

...

Running MIL default pipeline:  97%|█████████▋| 92/95 [00:01<00:00, 78.12 passes/s]

Running MIL default pipeline: 100%|██████████| 95/95 [00:01<00:00, 80.80 passes/s]

Running MIL backend_mlprogram pipeline:   0%|          | 0/12 [00:00<?, ? passes/s]

Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 438.76 passes/s]

saved CLIPImageEncoder.mlpackage


In [4]:
# Verify: PyTorch vs. the exported CoreML model, on a real sample image.
sample_image = Image.open(IMAGE_PATHS[0]).convert("RGB")
clip_inputs = clip_processor(images=sample_image, return_tensors="pt")
with torch.no_grad():
    torch_image_embed = clip_image_encoder(clip_inputs["pixel_values"]).numpy()[0]

coreml_out = clip_image_model.predict({"pixel_values": clip_preprocess_pil(sample_image)})
coreml_image_embed = coreml_out["image_embeds"][0]

cos_sim = np.dot(torch_image_embed, coreml_image_embed) / (
    np.linalg.norm(torch_image_embed) * np.linalg.norm(coreml_image_embed)
)
print(f"image encoder cosine sim (PyTorch fp32 vs CoreML fp16): {cos_sim:.5f}")


image encoder cosine sim (PyTorch fp32 vs CoreML fp16): 0.99317


### Text encoder

`torch.jit.trace` chokes on `CLIPModel.get_text_features`'s pooling step: it locates the EOS token via `input_ids.to(torch.int).argmax(dim=-1)` and gathers with `last_hidden_state[torch.arange(bsz), eos_idx]`. `coremltools`'s TorchScript frontend fails converting the resulting `aten::Int` op with `"only 0-dimensional arrays can be converted to Python scalars"`. Two fixes, both applied below:

1. **Pooling** &mdash; replace the fancy-indexing gather with an equivalent one-hot &middot; hidden-states contraction (`einsum`), which converts cleanly and is numerically identical.
2. **Causal mask** &mdash; the library helper builds the mask via `torch.full((seq_len, seq_len), ...)` where `seq_len` is read off the traced input tensor's `.shape`, which hits the same converter bug. Since CLIP's context length is a fixed constant (77) regardless of input, the mask is built from that literal Python int instead.

(fp16-unsafe `-inf` is also avoided: masked positions use `-1e4`, which safely saturates softmax to ~0 without overflowing in float16.)

In [5]:
from transformers.models.clip.modeling_clip import CLIPTextTransformer

NEG_INF = -1e4  # fp16-safe stand-in for -inf in additive attention masks


class ClipTextEncoder(torch.nn.Module):
    def __init__(self, clip, seq_len):
        super().__init__()
        text_model = clip.text_model
        self.embeddings = text_model.embeddings
        self.encoder = text_model.encoder
        self.final_layer_norm = text_model.final_layer_norm
        self.text_projection = clip.text_projection

        causal_mask = torch.triu(torch.full((seq_len, seq_len), NEG_INF), diagonal=1)
        self.register_buffer("causal_mask", causal_mask.unsqueeze(0).unsqueeze(0))
        self.seq_len = seq_len

    def forward(self, input_ids, attention_mask):
        hidden_states = self.embeddings(input_ids=input_ids)

        padding_mask = (1.0 - attention_mask[:, None, None, :].to(hidden_states.dtype)) * NEG_INF
        combined_mask = self.causal_mask + padding_mask

        encoder_outputs = self.encoder(
            inputs_embeds=hidden_states, attention_mask=combined_mask, causal_attention_mask=None
        )
        last_hidden_state = self.final_layer_norm(encoder_outputs.last_hidden_state)

        # EOS pooling via one-hot contraction instead of fancy indexing (see markdown above).
        eos_idx = input_ids.to(dtype=torch.int64).argmax(dim=-1)
        positions = torch.arange(self.seq_len, device=last_hidden_state.device).unsqueeze(0)
        one_hot = (positions == eos_idx.unsqueeze(1)).to(last_hidden_state.dtype)
        pooled_output = torch.einsum("bs,bsd->bd", one_hot, last_hidden_state)

        text_embeds = self.text_projection(pooled_output)
        return text_embeds / text_embeds.norm(p=2, dim=-1, keepdim=True)


clip_text_encoder = ClipTextEncoder(clip_model, CLIP_SEQ_LEN).eval()

sample_texts = ["a photo of a cat", "a photo of a dog", "a photo of a red sports car"]
sample_toks = clip_processor.tokenizer(
    sample_texts, padding="max_length", max_length=CLIP_SEQ_LEN, truncation=True, return_tensors="pt"
)

with torch.no_grad():
    ref_text_embeds = clip_model.get_text_features(**sample_toks)
    ref_text_embeds = ref_text_embeds / ref_text_embeds.norm(p=2, dim=-1, keepdim=True)
    our_text_embeds = clip_text_encoder(sample_toks["input_ids"], sample_toks["attention_mask"])
print("wrapper vs. get_text_features max abs diff:", (ref_text_embeds - our_text_embeds).abs().max().item())

with torch.no_grad():
    traced_text_encoder = torch.jit.trace(
        clip_text_encoder, (sample_toks["input_ids"], sample_toks["attention_mask"])
    )

clip_text_model = ct.convert(
    traced_text_encoder,
    inputs=[
        ct.TensorType(name="input_ids", shape=sample_toks["input_ids"].shape, dtype=np.int32),
        ct.TensorType(name="attention_mask", shape=sample_toks["attention_mask"].shape, dtype=np.int32),
    ],
    outputs=[ct.TensorType(name="text_embeds")],
    convert_to="mlprogram",
    minimum_deployment_target=ct.target.iOS16,
    compute_precision=ct.precision.FLOAT16,
)
clip_text_model.save(str(OUTPUT_DIR / "CLIPTextEncoder.mlpackage"))
print("saved CLIPTextEncoder.mlpackage")


wrapper vs. get_text_features max abs diff: 0.0


/Users/rafiabhista/Documents/code/apple/c3_agentic_ai/app-v1/trial_models/.venv/lib/python3.12/site-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/rafiabhista/Documents/code/apple/c3_agentic_ai/app-v1/trial_models/.venv/lib/python3.12/site-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/rafiabhista/Documents/code/apple/c3_agentic_ai/app-v1/trial_models/.venv/lib/python3.12/site-packages/transformers/models/clip/modeling_clip.py:282: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if seq_length > max_position_embedding:


Converting PyTorch Frontend ==> MIL Ops:   0%|          | 0/475 [00:00<?, ? ops/s]

Saving value type of int64 into a builtin type of int32, might lose precision!


Converting PyTorch Frontend ==> MIL Ops: 100%|█████████▉| 474/475 [00:00<00:00, 13093.91 ops/s]

...

/Users/rafiabhista/Documents/code/apple/c3_agentic_ai/app-v1/trial_models/.venv/lib/python3.12/site-packages/coremltools/converters/mil/mil/passes/defs/optimize_repeat_ops.py:433: RuntimeWarning: overflow encountered in cast
  max(cur_range.low, tmp_range.low), min(cur_range.high, tmp_range.high)
Running MIL default pipeline:  87%|████████▋ | 83/95 [00:00<00:00, 88.59 passes/s]

Running MIL default pipeline: 100%|██████████| 95/95 [00:01<00:00, 92.01 passes/s]

Running MIL backend_mlprogram pipeline:   0%|          | 0/12 [00:00<?, ? passes/s]

Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 433.04 passes/s]

saved CLIPTextEncoder.mlpackage


In [6]:
coreml_text_out = clip_text_model.predict({
    "input_ids": sample_toks["input_ids"].numpy().astype(np.int32),
    "attention_mask": sample_toks["attention_mask"].numpy().astype(np.int32),
})
ref_np = ref_text_embeds.numpy()
for i, text in enumerate(sample_texts):
    a, b = ref_np[i], coreml_text_out["text_embeds"][i]
    cos = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    print(f"{text!r:35s} cosine sim (PyTorch fp32 vs CoreML fp16): {cos:.5f}")


'a photo of a cat'                  cosine sim (PyTorch fp32 vs CoreML fp16): 0.99991
'a photo of a dog'                  cosine sim (PyTorch fp32 vs CoreML fp16): 0.99993
'a photo of a red sports car'       cosine sim (PyTorch fp32 vs CoreML fp16): 0.99977


### Sanity check: cross-modal retrieval using only the exported CoreML models

Embed every sample photo with `CLIPImageEncoder.mlpackage` and a few text queries with `CLIPTextEncoder.mlpackage`, then rank photos per query by cosine similarity &mdash; exactly the retrieval flow the iOS app will run.

In [7]:
retrieval_queries = ["a photo of a person", "a photo of food", "a photo of a landscape or building"]

image_embeds_by_path = {}
for path in IMAGE_PATHS:
    img = clip_preprocess_pil(Image.open(path).convert("RGB"))
    out = clip_image_model.predict({"pixel_values": img})
    image_embeds_by_path[path.name] = out["image_embeds"][0]

query_toks = clip_processor.tokenizer(
    retrieval_queries, padding="max_length", max_length=CLIP_SEQ_LEN, truncation=True, return_tensors="pt"
)
query_out = clip_text_model.predict({
    "input_ids": query_toks["input_ids"].numpy().astype(np.int32),
    "attention_mask": query_toks["attention_mask"].numpy().astype(np.int32),
})

for i, query in enumerate(retrieval_queries):
    q = query_out["text_embeds"][i]
    ranked = sorted(
        image_embeds_by_path.items(), key=lambda kv: -float(np.dot(q, kv[1]))
    )
    print(f"query: {query!r}")
    for name, emb in ranked:
        print(f"    {float(np.dot(q, emb)):.4f}  {name}")


query: 'a photo of a person'
    0.2317  IMG_4302.jpg
    0.1962  IMG_4277.JPG
    0.1867  IMG_4141.jpg
query: 'a photo of food'
    0.2556  IMG_4141.jpg
    0.2057  IMG_4302.jpg
    0.1962  IMG_4277.JPG
query: 'a photo of a landscape or building'
    0.2681  IMG_4302.jpg
    0.1762  IMG_4141.jpg
    0.1606  IMG_4277.JPG


## Part 2 — Florence-2-base (image captioning)

Loaded with `trust_remote_code=True` (Florence-2 ships its own `modeling_florence2.py`). `attn_implementation="eager"` is used so tracing goes through the plain (non-SDPA/flash) attention path.

In [8]:
from transformers import AutoModelForCausalLM, AutoProcessor

FLORENCE_MODEL_ID = "microsoft/Florence-2-base"
florence_model = AutoModelForCausalLM.from_pretrained(
    FLORENCE_MODEL_ID, trust_remote_code=True, attn_implementation="eager"
).eval()
florence_processor = AutoProcessor.from_pretrained(FLORENCE_MODEL_ID, trust_remote_code=True)
for p in florence_model.parameters():
    p.requires_grad_(False)

# The captioning task is fixed at export time -- Florence-2's processor maps this
# task token to a fixed natural-language prompt via `_construct_prompts` (e.g.
# "<CAPTION>" -> "What does the image describe?"), and that mapped prompt is what
# actually gets tokenized and embedded (never the literal task-token string --
# tokenizing "<CAPTION>" itself gives a different, untrained-for input that makes
# the decoder degenerate into garbage). This embedding never depends on the image.
# Swap TASK_PROMPT for "<DETAILED_CAPTION>" or "<MORE_DETAILED_CAPTION>" and re-run
# the export cells below for a different caption granularity.
TASK_PROMPT = "<CAPTION>"
mapped_prompt = florence_processor._construct_prompts([TASK_PROMPT])[0]
task_input_ids = florence_processor.tokenizer(text=mapped_prompt, return_tensors="pt")["input_ids"]
print(f"task prompt {TASK_PROMPT!r} is mapped to {mapped_prompt!r} before tokenization")

text_config = florence_model.config.text_config
FLORENCE_IMAGE_SIZE = florence_processor.image_processor.size["height"]
D_MODEL = text_config.d_model
DECODER_START_ID = text_config.decoder_start_token_id
PAD_ID = text_config.pad_token_id
EOS_ID = text_config.eos_token_id

print(f"image_size={FLORENCE_IMAGE_SIZE} d_model={D_MODEL} vocab={text_config.vocab_size} "
      f"enc_layers={text_config.encoder_layers} dec_layers={text_config.decoder_layers}")
print(f"tokenized mapped prompt -> input_ids {task_input_ids.tolist()} "
      f"-> {florence_processor.tokenizer.decode(task_input_ids[0])!r}")


task prompt '<CAPTION>' is mapped to 'What does the image describe?' before tokenization
image_size=768 d_model=768 vocab=51289 enc_layers=6 dec_layers=6
tokenized mapped prompt -> input_ids [[0, 2264, 473, 5, 2274, 6190, 116, 2]] -> '<s>What does the image describe?</s>'


In [9]:
# Reference caption straight from the original PyTorch model, for later comparison.
reference_image = Image.open(IMAGE_PATHS[0]).convert("RGB")
reference_inputs = florence_processor(text=TASK_PROMPT, images=reference_image, return_tensors="pt")

with torch.no_grad():
    reference_generated_ids = florence_model.generate(
        input_ids=reference_inputs["input_ids"],
        pixel_values=reference_inputs["pixel_values"],
        max_new_tokens=40,
        num_beams=1,
        do_sample=False,
    )
reference_text = florence_processor.batch_decode(reference_generated_ids, skip_special_tokens=False)[0]
reference_caption = florence_processor.post_process_generation(
    reference_text, task=TASK_PROMPT, image_size=(reference_image.width, reference_image.height)
)
print(f"[{IMAGE_PATHS[0].name}] reference caption: {reference_caption}")


/Users/rafiabhista/Documents/code/apple/c3_agentic_ai/app-v1/trial_models/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:679: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


[IMG_4141.jpg] reference caption: {'<CAPTION>': 'A table topped with lots of different types of sushi.'}


### Encoder: image + fixed task prompt &rarr; encoder hidden states

Wraps the model's own `_encode_image`, `_merge_input_ids_with_image_features`, and language-model encoder unchanged &mdash; `torch.export` handles their internal dynamic-shape mask/position-id construction that broke `torch.jit.trace` (see the intro cell). The only custom part is baking `TASK_PROMPT`'s token ids in as a buffer so the exported model's sole input is the image.

In [10]:
class FlorenceEncoder(torch.nn.Module):
    def __init__(self, model, task_input_ids):
        super().__init__()
        self.model = model
        self.register_buffer("task_input_ids", task_input_ids)

    def forward(self, pixel_values):
        image_features = self.model._encode_image(pixel_values)
        task_prefix_embeds = self.model.get_input_embeddings()(self.task_input_ids)
        merged_embeds, merged_attention_mask = self.model._merge_input_ids_with_image_features(
            image_features, task_prefix_embeds
        )
        return self.model.language_model.get_encoder()(
            inputs_embeds=merged_embeds, attention_mask=merged_attention_mask, return_dict=True
        ).last_hidden_state


florence_encoder_wrapper = FlorenceEncoder(florence_model, task_input_ids).eval()
pixel_values = reference_inputs["pixel_values"]

with torch.no_grad():
    reference_encoder_hidden_states = florence_encoder_wrapper(pixel_values)
    encoder_exported = torch.export.export(florence_encoder_wrapper, (pixel_values,))
    encoder_exported = encoder_exported.run_decompositions({})
print("encoder export + decompositions OK, output shape:", tuple(reference_encoder_hidden_states.shape))

florence_image_mean = florence_processor.image_processor.image_mean
florence_image_std = florence_processor.image_processor.image_std
florence_image_scale = 1.0 / (255.0 * float(np.mean(florence_image_std)))
florence_image_bias = [-m / s for m, s in zip(florence_image_mean, florence_image_std)]

florence_encoder_model = ct.convert(
    encoder_exported,
    inputs=[
        ct.ImageType(
            name="pixel_values",
            shape=pixel_values.shape,
            scale=florence_image_scale,
            bias=florence_image_bias,
            color_layout=ct.colorlayout.RGB,
        )
    ],
    outputs=[ct.TensorType(name="encoder_hidden_states")],
    convert_to="mlprogram",
    minimum_deployment_target=ct.target.iOS16,
    compute_precision=ct.precision.FLOAT16,
)
florence_encoder_model.save(str(OUTPUT_DIR / "FlorenceEncoder.mlpackage"))
print("saved FlorenceEncoder.mlpackage")


encoder export + decompositions OK, output shape: (1, 585, 768)


Converting PyTorch Frontend ==> MIL Ops:   0%|          | 0/1199 [00:00<?, ? ops/s]

Converting PyTorch Frontend ==> MIL Ops:  65%|██████▍   | 775/1199 [00:00<00:00, 7746.66 ops/s]

Saving value type of int64 into a builtin type of int32, might lose precision!


Saving value type of int64 into a builtin type of int32, might lose precision!


...

Running MIL default pipeline: 100%|██████████| 95/95 [00:02<00:00, 32.22 passes/s]

Running MIL backend_mlprogram pipeline:   0%|          | 0/12 [00:00<?, ? passes/s]

Running MIL backend_mlprogram pipeline:  83%|████████▎ | 10/12 [00:00<00:00, 98.91 passes/s]

Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 102.98 passes/s]

saved FlorenceEncoder.mlpackage


In [11]:
coreml_encoder_out = florence_encoder_model.predict(
    {"pixel_values": reference_image.resize((FLORENCE_IMAGE_SIZE, FLORENCE_IMAGE_SIZE))}
)["encoder_hidden_states"][0]
ref_enc_np = reference_encoder_hidden_states.numpy()[0]
diff = np.abs(coreml_encoder_out - ref_enc_np)
cos = np.sum(coreml_encoder_out * ref_enc_np) / (
    np.linalg.norm(coreml_encoder_out) * np.linalg.norm(ref_enc_np)
)
print(f"encoder hidden states: mean abs diff={diff.mean():.4f} max abs diff={diff.max():.4f} cosine sim={cos:.5f}")


encoder hidden states: mean abs diff=0.0655 max abs diff=32.4506 cosine sim=0.99648


### Decoder step: tokens-so-far + encoder hidden states &rarr; next-token logits

No KV-cache (see intro cell): `decoder_input_ids` is a fixed-length buffer (`MAX_CAPTION_TOKENS`), right-padded, and the *whole* prefix is recomputed on every call. Causal masking already guarantees a query at position *i* only sees keys at positions &le; *i*, so padding placed after the real tokens is simply never attended to by anything that matters &mdash; iOS just reads the logits at index `current_length - 1` for the next token.

In [12]:
MAX_CAPTION_TOKENS = 40


class FlorenceDecoderStep(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.decoder = model.language_model.get_decoder()
        self.lm_head = model.language_model.lm_head
        self.register_buffer("final_logits_bias", model.language_model.final_logits_bias)

    def forward(self, decoder_input_ids, encoder_hidden_states):
        decoder_out = self.decoder(
            input_ids=decoder_input_ids,
            encoder_hidden_states=encoder_hidden_states,
            past_key_values=None,
            use_cache=False,
            return_dict=True,
        )
        return self.lm_head(decoder_out.last_hidden_state) + self.final_logits_bias


florence_decoder_wrapper = FlorenceDecoderStep(florence_model).eval()

dummy_decoder_ids = torch.full((1, MAX_CAPTION_TOKENS), PAD_ID, dtype=torch.long)
dummy_decoder_ids[0, 0] = DECODER_START_ID

with torch.no_grad():
    decoder_exported = torch.export.export(
        florence_decoder_wrapper, (dummy_decoder_ids, reference_encoder_hidden_states)
    )
    decoder_exported = decoder_exported.run_decompositions({})
print("decoder export + decompositions OK")

florence_decoder_model = ct.convert(
    decoder_exported,
    inputs=[
        ct.TensorType(name="decoder_input_ids", shape=dummy_decoder_ids.shape, dtype=np.int32),
        ct.TensorType(name="encoder_hidden_states", shape=reference_encoder_hidden_states.shape),
    ],
    outputs=[ct.TensorType(name="logits")],
    convert_to="mlprogram",
    minimum_deployment_target=ct.target.iOS16,
    compute_precision=ct.precision.FLOAT16,
)
florence_decoder_model.save(str(OUTPUT_DIR / "FlorenceDecoderStep.mlpackage"))
print("saved FlorenceDecoderStep.mlpackage")


decoder export + decompositions OK


Converting PyTorch Frontend ==> MIL Ops:   0%|          | 0/279 [00:00<?, ? ops/s]

Core ML embedding (gather) layer does not support any inputs besides the weights and indices. Those given will be ignored.


Saving value type of int64 into a builtin type of int32, might lose precision!


Saving value type of int64 into a builtin type of int32, might lose precision!


...

Running MIL default pipeline:  97%|█████████▋| 92/95 [00:01<00:00, 52.51 passes/s]

Running MIL default pipeline: 100%|██████████| 95/95 [00:01<00:00, 53.64 passes/s]

Running MIL backend_mlprogram pipeline:   0%|          | 0/12 [00:00<?, ? passes/s]

Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 515.57 passes/s]

saved FlorenceDecoderStep.mlpackage


### End-to-end greedy caption generation using only the exported CoreML models

Encoder runs once; the decoder step runs in a plain greedy loop (argmax, append, repeat until `</s>` or `MAX_CAPTION_TOKENS`) &mdash; exactly the loop the iOS app implements in Swift.

In [13]:
def generate_caption_with_coreml(image, task_prompt=TASK_PROMPT):
    inputs = florence_processor(text=task_prompt, images=image, return_tensors="pt")
    pixel_values_resized = image.resize((FLORENCE_IMAGE_SIZE, FLORENCE_IMAGE_SIZE))

    encoder_hidden_states = florence_encoder_model.predict({"pixel_values": pixel_values_resized})[
        "encoder_hidden_states"
    ].astype(np.float32)

    decoder_ids = np.full((1, MAX_CAPTION_TOKENS), PAD_ID, dtype=np.int32)
    decoder_ids[0, 0] = DECODER_START_ID
    generated = [DECODER_START_ID]

    for step in range(MAX_CAPTION_TOKENS - 1):
        logits = florence_decoder_model.predict(
            {"decoder_input_ids": decoder_ids, "encoder_hidden_states": encoder_hidden_states}
        )["logits"]
        next_id = int(np.argmax(logits[0, len(generated) - 1]))
        generated.append(next_id)
        if next_id == EOS_ID:
            break
        decoder_ids[0, len(generated) - 1] = next_id

    text = florence_processor.tokenizer.decode(generated, skip_special_tokens=False)
    return florence_processor.post_process_generation(
        text, task=task_prompt, image_size=(image.width, image.height)
    )


for path in IMAGE_PATHS:
    img = Image.open(path).convert("RGB")
    caption = generate_caption_with_coreml(img)
    print(f"[{path.name}] {caption}")

print()
print(f"(reference, PyTorch generate() on {IMAGE_PATHS[0].name}): {reference_caption}")


[IMG_4141.jpg] {'<CAPTION>': 'A table topped with lots of different types of sushi.'}


[IMG_4277.JPG] {'<CAPTION>': 'A group of people standing in an elevator.'}


[IMG_4302.jpg] {'<CAPTION>': 'A view of a tall building with a cloudy sky in the background.'}

(reference, PyTorch generate() on IMG_4141.jpg): {'<CAPTION>': 'A table topped with lots of different types of sushi.'}


## Tokenizer assets for on-device use

The exported models operate on token ids and pixel buffers, not raw text. iOS needs the CLIP BPE vocab (to tokenize retrieval queries) and the Florence-2 tokenizer (to decode generated ids back to text) &mdash; e.g. via Hugging Face's [swift-transformers](https://github.com/huggingface/swift-transformers) `Tokenizers` package, which loads these files directly.

In [14]:
clip_tokenizer_dir = OUTPUT_DIR / "clip_tokenizer"
clip_processor.tokenizer.save_pretrained(clip_tokenizer_dir)

florence_tokenizer_dir = OUTPUT_DIR / "florence_tokenizer"
florence_processor.tokenizer.save_pretrained(florence_tokenizer_dir)

print("CLIP tokenizer files:", sorted(p.name for p in clip_tokenizer_dir.iterdir()))
print("Florence tokenizer files:", sorted(p.name for p in florence_tokenizer_dir.iterdir()))


CLIP tokenizer files: ['merges.txt', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']
Florence tokenizer files: ['added_tokens.json', 'merges.txt', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']


## Summary

In [15]:
def dir_size_mb(path):
    if path.is_file():
        return path.stat().st_size / 1e6
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e6


print(f"{'artifact':32s} {'size (MB)':>10s}")
for item in sorted(OUTPUT_DIR.iterdir()):
    print(f"{item.name:32s} {dir_size_mb(item):10.1f}")


artifact                          size (MB)
CLIPImageEncoder.mlpackage            175.9
CLIPTextEncoder.mlpackage             127.0
FlorenceDecoderStep.mlpackage         192.6
FlorenceEncoder.mlpackage             270.5
clip_tokenizer                          5.0
florence_tokenizer                      5.4


### iOS integration notes

- **Retrieval (CLIP):** `CLIPImageEncoder.mlpackage` takes a 224&times;224 RGB `CVPixelBuffer` (`Vision`/`CoreImage` resize is fine &mdash; normalization is baked in) and returns a 512-d embedding; embed the photo library once and cache the embeddings. `CLIPTextEncoder.mlpackage` takes `input_ids`/`attention_mask` of fixed shape `[1, 77]` (BPE-tokenize and pad with the CLIP tokenizer's own pad/eos convention) and returns a 512-d embedding in the same space. Rank photos by dot product (both sides are already L2-normalized).
- **Captioning (Florence-2):** `FlorenceEncoder.mlpackage` takes a 768&times;768 RGB buffer and returns `[1, 583, 768]` encoder hidden states (run once per photo). `FlorenceDecoderStep.mlpackage` takes those hidden states plus a `[1, 40]` `decoder_input_ids` buffer (start with `[2, 1, 1, ..., 1]` &mdash; index 0 is `decoder_start_token_id`, rest is `pad_token_id`) and returns `[1, 40, vocab]` logits; read row `current_length - 1`, argmax (or sample) for the next token, write it into the next pad slot, repeat until `eos_token_id` (2) or the buffer fills. Decode the collected ids with the Florence tokenizer and pass through `processor.post_process_generation`-equivalent cleanup (mainly stripping `<s>`/`</s>`).
- **Caption granularity / different task:** re-run the "Florence-2-base" section with `TASK_PROMPT` set to `"<DETAILED_CAPTION>"` or `"<MORE_DETAILED_CAPTION>"` (or any other Florence-2 task token) to bake a different fixed prompt into a new `FlorenceEncoder.mlpackage` &mdash; the decoder step model is unaffected and can be reused as-is.
- **Precision:** both models are exported `FLOAT16` (`minimum_deployment_target=.iOS16`), validated above at cosine similarity &ge; 0.99 against the fp32 PyTorch reference (text encoders are closer to 0.9999; image-heavy encoders, which pass through many more fp16-quantized layers, land around 0.993-0.996). If retrieval/caption quality regresses on-device, re-run with `compute_precision=ct.precision.FLOAT32` for a larger but more precise model.
